# Data-Driven Insights for Sustainable Development Goals
### EPA112A Final Assignment 2026 - Group 17

| Name | Student number |
|---|---|
| Hazel Lim | 5697158 |
| Woobie Mizobuchi Reeh | 6593720 |

**Group number:** 17

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import wbdata

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

DATA_DIR = Path("data")
START_YEAR, END_YEAR = 1960, 2025

## 1. Introduction

### Problem statement

Maternal and child mortality remain major public-health challenges, and their burdens differ considerably between countries. National wealth is an important part of this inequality: wealthier countries generally have more resources to support healthcare, sanitation, and other conditions associated with survival. Yet income alone may not capture how political institutions and social opportunities affect health outcomes.

Women's political empowerment offers another perspective. When women can exercise civil liberties, participate in civil society, and engage in political decision-making, their interests may be better represented in public life. However, whether countries with greater women's political empowerment also experience lower maternal and child mortality (and whether that relationship persists after accounting for national income) is an empirical question.

Our project is framed primarily by **SDG 3: Good Health and Well-being**, particularly **Target 3.1** (reducing maternal mortality) and **Target 3.2** (ending preventable deaths of newborns and children under five). It also connects to **SDG 5: Gender Equality**, especially women's participation in decision-making. The analysis focuses on political empowerment rather than treating it as a complete measure of gender equality.

### Main research question

**To what extent is women's political empowerment associated with maternal and under-five mortality across countries, after accounting for GDP per capita?**

### Subquestions

1. How is women's political empowerment associated with maternal mortality and under-five mortality across countries in 2022?
2. How do these relationships compare with the associations between GDP per capita and mortality?
3. Do the observed patterns remain similar in 2023, and does the association with empowerment persist when GDP per capita is included in a statistical model?

### Study design

We use 2022 as the primary cross-section and 2023 as a sensitivity check. The unit of observation is a country-year. The analysis is observational and descriptive/associational; it cannot establish that women's political empowerment causes changes in mortality. All countries with the necessary data are eligible, rather than restricting the sample to a particular income group.

## 2. Datasets

We combine two independent public data sources:

| Source | Indicator | Code / column | Unit | Role |
|---|---|---|---|---|
| World Bank, World Development Indicators (WDI) | Maternal mortality ratio | `SH.STA.MMRT` | Maternal deaths per 100,000 live births | Outcome 1 |
| World Bank, WDI | Under-five mortality rate | `SH.DYN.MORT` | Deaths before age five per 1,000 live births | Outcome 2 |
| World Bank, WDI | GDP per capita, PPP (constant international $) | `NY.GDP.PCAP.PP.KD` | International dollars per person | Income control |
| V-Dem, Country-Year Core v16 | Women's Political Empowerment Index | `v2x_gender` | Index from 0 to 1; higher = greater empowerment | Main explanatory variable |

**Data provenance:** [World Bank WDI](https://data.worldbank.org/) and [V-Dem dataset](https://www.v-dem.net/data/the-v-dem-dataset/). The V-Dem index measures women's **political empowerment** through civil liberties, civil society participation, and political participation. It should not be interpreted as a comprehensive index of gender equality.

The World Bank downloads contain countries **and** regional/income aggregates, which must be excluded. World Bank data arrive in **wide format** (one column per year), whereas V-Dem arrives in **long format** (one row per country-year). We merge by the three-letter country code and year, not by country names, because names can differ across sources.

**Temporal availability:** The source files contain many historical years. We retain 2000–2024 during preparation, but maternal mortality is unavailable for 2024 in the supplied release. Therefore, the complete-case analysis covers 2000–2023; 2022 and 2023 are used for the main comparison.

### Input files

- `API_SH.STA.MMRT_DS2_EN_csv_v2_2368.csv`
- `API_SH.DYN.MORT_DS2_EN_csv_v2_2145.csv`
- `API_NY.GDP.PCAP.PP.KD_DS2_EN_csv_v2_2481.csv`
- `V-Dem-CY-Core-v16.csv`

In [ ]:
%pip install pycountry

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pycountry

In [ ]:
# Use the same folder structure for all group members.
DATA_DIR = Path("data/raw_data")
OUTPUT_DIR = Path("data/processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WB_FILES = {
    "maternal": "API_SH.STA.MMRT_DS2_EN_csv_v2_2368.csv",
    "under5": "API_SH.DYN.MORT_DS2_EN_csv_v2_2145.csv",
    "gdp_ppp": "API_NY.GDP.PCAP.PP.KD_DS2_EN_csv_v2_2481.csv",
}
VDEM_FILE = "V-Dem-CY-Core-v16.csv"
YEARS = list(range(2000, 2025))
MAIN_VARIABLES = ["v2x_gender", "maternal", "under5", "gdp_ppp"]

for filename in list(WB_FILES.values()) + [VDEM_FILE]:
    if not (DATA_DIR / filename).exists():
        raise FileNotFoundError(f"Missing original input: {DATA_DIR / filename}")

print("All four source files found.")

## 3. Preprocessing

The following steps are performed directly in Python:

1. **Exclude World Bank aggregates:** keep valid ISO three-letter country codes (plus `XKX`, used for Kosovo). We use `pycountry` for this identification rather than treating every World Bank row as a country.
2. **Reshape World Bank indicators:** convert year columns into a `year` column using `pandas.melt()`.
3. **Standardize merge keys:** rename the country-code fields to `country_code`, convert `year` to an integer, and convert measurements to numeric values.
4. **Select V-Dem fields:** retain the women's political empowerment index and its three component indices; restrict to the same year range.
5. **Merge:** perform one-to-one outer merges on `country_code` and `year` to preserve coverage information, including rows missing from a source.
6. **Audit:** check duplicate keys, missing values, and country coverage by year. We do **not** fill missing values with zero or impute missing mortality measurements.
7. **Create analysis samples:** retain only complete observations for all four main variables in 2022 and 2023. This selection can introduce coverage bias, which we will discuss later.

In [ ]:
# Country codes: prevent regional and income-group aggregates entering the analysis.
valid_country_codes = {country.alpha_3 for country in pycountry.countries}
valid_country_codes.add("XKX")  # Kosovo


def load_world_bank(filename, variable_name):
    """Read one original WDI CSV and return country-year observations."""
    wide = pd.read_csv(DATA_DIR / filename, skiprows=4, low_memory=False)
    wide = wide.loc[:, ~wide.columns.str.startswith("Unnamed:")].copy()
    wide = wide[wide["Country Code"].isin(valid_country_codes)].copy()

    year_columns = [str(year) for year in YEARS]
    if not set(year_columns).issubset(wide.columns):
        raise ValueError(f"Expected year columns missing from {filename}")

    long = wide.melt(
        id_vars=["Country Name", "Country Code"],
        value_vars=year_columns,
        var_name="year",
        value_name=variable_name,
    )
    long = long.rename(columns={
        "Country Name": "wb_country_name",
        "Country Code": "country_code",
    })
    long["year"] = long["year"].astype(int)
    long[variable_name] = pd.to_numeric(long[variable_name], errors="coerce")

    if long.duplicated(["country_code", "year"]).any():
        raise ValueError(f"Duplicate country-year keys in {filename}")
    return long


world_bank = {}
for variable, filename in WB_FILES.items():
    world_bank[variable] = load_world_bank(filename, variable)
    print(variable, world_bank[variable].shape,
          "nonmissing:", world_bank[variable][variable].notna().sum())

In [ ]:
# Standardize V-Dem keys and retain only the needed variables.
vdem = vdem_raw.rename(columns={"country_text_id": "country_code"}).copy()
vdem = vdem[
    vdem["country_code"].isin(valid_country_codes)
    & vdem["year"].between(min(YEARS), max(YEARS))
].copy()
vdem["year"] = vdem["year"].astype(int)

if vdem.duplicated(["country_code", "year"]).any():
    raise ValueError("Duplicate V-Dem country-year keys")

print("V-Dem countries:", vdem["country_code"].nunique())
print("V-Dem observations, 2000–2024:", len(vdem))

In [ ]:
# Outer merge keeps all available country-year rows for missingness auditing.
panel = vdem.copy()
for variable in ["maternal", "under5", "gdp_ppp"]:
    wb_values = world_bank[variable][["country_code", "year", variable]]
    panel = panel.merge(wb_values, on=["country_code", "year"],
                        how="outer", validate="one_to_one")

# Fill country names for countries appearing only in World Bank files.
wb_names = pd.concat([
    frame[["country_code", "wb_country_name"]]
    for frame in world_bank.values()
], ignore_index=True).drop_duplicates(subset="country_code")
panel = panel.merge(wb_names, on="country_code", how="left",
                    validate="many_to_one")
panel["country_name"] = panel["country_name"].fillna(panel["wb_country_name"])
panel = panel.drop(columns="wb_country_name")
panel = panel.sort_values(["country_code", "year"]).reset_index(drop=True)

panel["complete_case"] = panel[MAIN_VARIABLES].notna().all(axis=1)
panel["complete_core"] = panel[["v2x_gender", "maternal", "under5"]].notna().all(axis=1)

assert not panel.duplicated(["country_code", "year"]).any()
assert panel.loc[panel["complete_case"], MAIN_VARIABLES].notna().all().all()
print("Merged country-year rows:", len(panel))
print("Countries in merged panel:", panel["country_code"].nunique())
display(panel.head())

In [ ]:
# Audit coverage and missingness by year.
coverage_by_year = panel.groupby("year").agg(
    country_rows=("country_code", "size"),
    complete_cases=("complete_case", "sum"),
    **{f"{col}_available": (col, "count") for col in MAIN_VARIABLES},
).reset_index()

for col in MAIN_VARIABLES:
    coverage_by_year[f"{col}_missing_pct"] = (
        100 * (1 - coverage_by_year[f"{col}_available"] /
                 coverage_by_year["country_rows"])
    ).round(1)

display(coverage_by_year.tail(6))

# Missingness in the two primary cross-sectional samples before complete-case filtering.
for year in [2022, 2023]:
    subset = panel[panel["year"] == year]
    print(f"\n{year}: {len(subset)} country rows, "
          f"{subset['complete_case'].sum()} complete cases")
    display(subset[MAIN_VARIABLES].isna().sum().rename("missing_count").to_frame())

In [ ]:
# Country-level completeness across the full 2000–2024 preparation window.
coverage_by_country = panel.groupby(["country_code", "country_name"],
                                    dropna=False).agg(
    years_observed=("year", "size"),
    complete_years=("complete_case", "sum"),
    **{f"{col}_available": (col, "count") for col in MAIN_VARIABLES},
).reset_index()

# Create the primary and sensitivity-check samples.
data_2022 = panel[(panel["year"] == 2022) & panel["complete_case"]].copy()
data_2023 = panel[(panel["year"] == 2023) & panel["complete_case"]].copy()
complete_panel = panel[(panel["year"] <= 2023) & panel["complete_case"]].copy()

assert len(data_2022) == 167, "Unexpected 2022 count; check source versions"
assert len(data_2023) == 166, "Unexpected 2023 count; check source versions"
assert len(complete_panel) == 3974, "Unexpected panel count; check source versions"

print("2022 analysis sample:", len(data_2022), "countries")
print("2023 analysis sample:", len(data_2023), "countries")
print("Complete 2000–2023 observations:", len(complete_panel))

# Save outputs for subsequent EDA and regression cells.
panel.to_csv(OUTPUT_DIR / "women_clean_merged_panel_2000_2024.csv", index=False)
complete_panel.to_csv(OUTPUT_DIR / "women_complete_panel_2000_2023.csv", index=False)
data_2022.to_csv(OUTPUT_DIR / "women_2022.csv", index=False)
data_2023.to_csv(OUTPUT_DIR / "women_2023.csv", index=False)
coverage_by_year.to_csv(OUTPUT_DIR / "coverage_by_year.csv", index=False)
coverage_by_country.to_csv(OUTPUT_DIR / "coverage_by_country.csv", index=False)
print("Processed files saved to", OUTPUT_DIR)

### Preprocessing decisions and limitations

The cleaned dataset retains original numerical values; missing observations are not replaced with zeros or estimated values. The primary 2022 analysis includes **167 countries**, and the 2023 comparison includes **166 countries**, each with nonmissing values for all four main indicators. The full merged 2000–2024 file retains incomplete observations so that missingness remains visible. The complete-case 2000–2023 panel contains **3,974 country-year observations**.

Using an outer merge makes source coverage auditable but does not make all countries comparable: only complete-case rows enter the subsequent models. Country availability may be systematically related to income, governance, or health conditions, so the sample is not necessarily representative of every country. The 2022 and 2023 samples are not perfectly identical; comparisons between them may reflect changes in sample composition as well as indicator values.

We also retain extreme but valid-looking values at this stage. Potential outliers will be investigated in the EDA section rather than automatically removed. Later analyses will use logarithmic transformations for strictly positive mortality rates and GDP per capita to address skewness. No causal interpretation is made from these observational cross-country comparisons.

## 4. Data analysis

## 5. Visualization

## 6. Discussion

## 7. Conclusion

## 8. Reflection

## 9. AI Use Statement